# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 53d9c906-c75d-4d0e-ae8b-d30981… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ 7af8e60b-fe27-4fc4-a098-0b7817… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-01 ┆ claude-hai ┆ 567.0      ┆ 1121.0    ┆ 11888206  ┆ 5916990   ┆ 0.04936   ┆ 33.178525 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## 📝 Báo cáo phân tích & Trả lời câu hỏi Checkpoint 4 (Rubric Criteria)

### 1. Bước Dedup ở tầng Silver giải quyết vấn đề gì?
- Trong kiến trúc dữ liệu hiện đại, việc thu thập sự kiện (LLM Observability Logs, CDC, IoT Telemetry) qua các hệ thống streaming message brokers (Kafka, Kinesis, RabbitMQ, HTTP Webhooks) thường hoạt động theo cơ chế **At-Least-Once Delivery** (đảm bảo không mất dữ liệu bằng cách thử lại khi có lỗi mạng).
- Tầng **Bronze** là tầng hạ cánh nguyên bản (Raw Landing Zone, append-only), lưu trữ mọi payload thô nhận được kèm retry logs mà không can thiệp. Vì vậy, Bronze luôn chứa các bản ghi bị trùng lặp (trong bài lab đã cố ý inject 9,948 bản ghi trùng `request_id`).
- Bước **Deduplication (Dedup)** ở tầng **Silver** sử dụng khóa nghiệp vụ `request_id` kết hợp với window function (`ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts)`) để loại bỏ hoàn toàn các bản ghi dư thừa, giải quyết 2 bài toán lớn:
  1. Ngăn chặn hiện tượng tính toán khống (double counting) các chỉ số chi phí token, số lượt request, và doanh thu.
  2. Chuẩn hóa dữ liệu sạch (Clean & Curated Single Source of Truth) phục vụ cho các phân tích nghiệp vụ và huấn luyện mô hình.

### 2. Vì sao Dashboard nên đọc dữ liệu từ tầng Gold thay vì Silver?
- **Tối ưu hóa hiệu năng truy vấn và độ trễ (Query Performance & Latency)**: Tầng Silver chứa hàng trăm nghìn hoặc hàng triệu dòng chi tiết từng lượt gọi API (fine-grained event logs). Nếu dashboard trực tiếp truy vấn Silver, mỗi lần người dùng mở hoặc làm mới trang, hệ thống phải quét hàng chục GB dữ liệu, thực hiện parse và tính toán quantile (p50, p95 latency) trên toàn bộ tập dữ liệu, dẫn đến độ trễ cao (hàng chục giây đến vài phút).
- **Tiết kiệm tài nguyên tính toán và chi phí FinOps**: Tầng Gold đã được tính toán tổng hợp trước (pre-aggregated) theo các chiều nghiệp vụ (`date`, `model`), giảm kích thước dữ liệu từ hàng triệu dòng xuống chỉ còn vài chục dòng. Dashboard đọc Gold chỉ mất vài mili-giây và tiêu tốn lượng I/O cực nhỏ.
- **Thống nhất logic nghiệp vụ (Standardized Business Metrics)**: Gold định nghĩa sẵn công thức chuẩn duy nhất về SLA độ trễ, tỷ lệ lỗi (error rate), và chi phí token USD, tránh trường hợp mỗi dashboard tự viết query tính toán dẫn đến sai lệch số liệu báo cáo giữa các phòng ban.

### 3. Đánh giá tính hợp lý của cách tính Error Rate và Chi phí (Cost USD)
- **Công thức Error Rate**: Được tính bằng `AVG(CASE WHEN status <> 'ok' THEN 1.0 ELSE 0.0 END)`. Đây là công thức chuẩn xác, phản ánh tỷ lệ phần trăm các cuộc gọi API có trạng thái lỗi trên tổng số cuộc gọi trong ngày của model đó, giá trị kết quả luôn nằm trong đoạn `[0.0, 1.0]`.
- **Công thức Chi phí (Cost USD)**: Tính theo đơn giá token chuẩn: `(SUM(prompt_tokens) * price_in + SUM(completion_tokens) * price_out) / 1,000,000`. Dữ liệu đầu vào có token count luôn dương, đơn giá completion cao hơn prompt phù hợp với thực tế vận hành các mô hình ngôn ngữ lớn (Claude, GPT), cho ra `cost_usd` dương và phản ánh chính xác cấu trúc chi phí vận hành.
